<a href="https://colab.research.google.com/github/Serragem/ModelagemVQE/blob/main/TwoLocalH2%2C4Qb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install qiskit -U
!pip install qiskit_aer -U
!pip install qiskit-ibm-runtime -U

!pip install matplotlib
!pip install pylatexenc
!pip install qiskit-nature
!pip install pyscf

import qiskit
qiskit.__version__

'2.4.1'

In [2]:
# Qiskit: métodos básicos
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister
from qiskit import transpile
from qiskit.visualization import plot_histogram, array_to_latex, plot_state_city
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from scipy.optimize import minimize
import numpy as np
import matplotlib.pyplot as plt
from qiskit.circuit.library import TwoLocal
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator
from scipy.optimize import minimize

## TESTE DO TWOLOCAL COM H2, 4 QUIBITS

In [ ]:
#from qiskit.circuit.library import TwoLocal

#ansatz_H2_direto = TwoLocal(
    #num_qubits=4,
    ##num_qubits=4: Alocado para representar os 4 spin-orbitais da base STO-3G
    #rotation_blocks=['ry', 'rz'],
    ##rotation_blocks=['ry', 'rz']: Isso ocorre porque a função de onda do estado fundamental de moléculas de camada fechada (H2) geralmente possui coeficientes reais. A porta Ry explora amplitudes puramente reais, enquanto a Rz permite o ajuste fino de fases, caso necessário
    #entanglement_blocks='cx',
    ##entanglement_blocks='cx': A porta CNOT (CX) é a porta de emaranhamento nativa da grande maioria dos hardwares supercondutores (como os da IBM), sendo preferida sobre a CZ em topologias maiores.
    #entanglement='linear',
    ##entanglement='linear': Este é um parâmetro novo e crucial. Como temos mais de 2 qubits, precisamos definir como eles se conectam. O padrão 'linear' aplica as portas CX apenas entre qubits vizinhos (0-1, 1-2, 2-3). Isso é o chamado hardware-efficient, pois evita a necessidade de portas SWAP custosas em processadores reais que possuem conectividade linear. Outra opção seria 'full' (conecta todos com todos), que aumenta a expressividade, mas gera circuitos muito profundos.
    #reps=2
    ##reps=2: Como o espaço de busca é maior (2^4 = 16 dimensões), aumentamos as repetições para 2. O circuito alternará: Rotação $\rightarrow$ Emaranhamento $\rightarrow$ Rotação $\rightarrow$ Emaranhamento $\rightarrow$ Rotação final.
#)
#

## TESTE COMPLETO PARA H2 COM 4 QUIBITS

## CRIAÇÃO DO DRIVER COM APOIO DA PySCFDriver

In [3]:
import numpy as np
from scipy.optimize import minimize
from qiskit.circuit.library import TwoLocal
from qiskit.primitives import StatevectorEstimator
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper

# Distância teórica de eq
distancia_H2 = 0.735

driver = PySCFDriver(
    # PySCFDriver É a classe que instancia o motor clássico de química quântica (neste caso, a biblioteca PySCF).
    atom=f"H 0 0 0; H 0 0 {distancia_H2}",
    #Define a geometria espacial da molécula. A sintaxe é uma string contendo o símbolo do elemento e suas coordenadas cartesianas X, Y e Z. Aqui, fixamos um hidrogênio na origem e o outro no eixo Z, separados pela variável distancia_H2
    basis="sto3g",
    #basis="sto3g": Especifica o conjunto de base de orbitais atômicos a ser utilizado. STO-3G (Slater-Type Orbitals aproximados por 3 funções Gaussianas) é a base mais fundamental. Para o H2, isso gera 2 orbitais espaciais, o que resulta em 4 spin-orbitais (logo, 4 qubits no mapeamento direto).
    charge=0,
    #charge=0: Define a carga líquida da molécula. Como estamos modelando a molécula de hidrogênio em seu estado neutro, o valor é nulo.
    spin=0,
    #spin=0: Refere-se à diferença entre o número de elétrons alpha (spin up) e beta (spin down), formalmente equivalente a 2S, onde S é o spin total do sistema. O estado fundamental do H2 é um singleto (elétrons emparelhados), portanto, o valor é zero.
    unit=DistanceUnit.ANGSTROM,
    #unidade
)

# Executa o driver para obter as propriedades eletrônicas clássicas
problema_eletronico = driver.run()

## MAPEAMENTO FERMIÔNICO PARA QUIBITS

In [4]:
hamiltoniano_fermionico = problema_eletronico.hamiltonian.second_q_op()
# second_q_op(): Método que extrai o Hamiltoniano da molécula em sua forma de segunda quantização a partir dos dados do driver. Ele converte as propriedades eletrônicas clássicas em uma expressão que utiliza operadores de criação (a) e aniquilação (a*) de férmions.

mapper = JordanWignerMapper()
# JordanWignerMapper(): Instancia o objeto encarregado da tradução algébrica do sistema.  O mapeamento de Jordan-Wigner estabelece uma relação biunívoca entre a ocupação dos spin-orbitais (vazio ou ocupado) e os estados |0> e |1> dos qubits. Crucialmente, ele introduz cadeias de matrizes de Pauli Z no circuito para garantir que o Princípio de Exclusão de Pauli (anticomutação fermiônica) seja matematicamente respeitado pelos qubits que, por natureza, comutam.
hamiltoniano_qubit = mapper.map(hamiltoniano_fermionico)

print(f"Dimensionalidade: O Hamiltoniano foi mapeado para {hamiltoniano_qubit.num_qubits} qubits.")


Dimensionalidade: O Hamiltoniano foi mapeado para 4 qubits.


## CONSTRUÇÃO DO ANSATZ COM TWOLOCAL

In [5]:
ansatz = TwoLocal(
    num_qubits=hamiltoniano_qubit.num_qubits,
    rotation_blocks=['ry', 'rz'],
    entanglement_blocks='cx',
    entanglement='linear',
    reps=2
)


/tmp/ipykernel_12783/3566506279.py:1: DeprecationWarning: The class ``qiskit.circuit.library.n_local.two_local.TwoLocal`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.n_local instead.
  ansatz = TwoLocal(


## CONFIGURAÇÃO DO ESTIMADOR E FUNÇÃO OBJETIVO

In [6]:
estimator = StatevectorEstimator()
ponto_inicial = np.zeros(ansatz.num_parameters)
# nenhum algoritmo iterativo pode começar a procurar o mínimo sem um ponto de partida definido. É imperativo fornecer ao otimizador uma coordenada inicial no espaço de parâmetros para que ele dê o "primeiro passo". É exatamente esta a função da variável ponto_inicial.
# a função zeros da biblioteca clássica NumPy (np) cria um array (vetor) dimensional preenchido inteiramente com o valor 0.0. O tamanho desse vetor é ditado pelo argumento que acabamos de analisar.
# resultado prático: Se o seu ansatz possuir 8 parâmetros, ponto_inicial será o vetor dinâmico [0., 0., 0., 0., 0., 0., 0., 0.]

def funcao_objetivo(parametros, ansatz_circ, hamiltoniano, estimador):
    # estimator.run([(ansatz_circ, hamiltoniano, parametros)]): Esta é a sintaxe de execução do StatevectorEstimator nas Primitivas V2 do Qiskit. Ele exige como argumento uma lista de tuplas estruturada rigorosamente na ordem: (circuito de tentativa, operador observável, valores atuais dos parâmetros)
    job = estimador.run([(ansatz_circ, hamiltoniano, parametros)])
    resultado = job.result()[0]

    # Retorna o valor esperado (evs - expectation values)
    return resultado.data.evs


## OTIMIZAÇÃO CLASSICA VQE

In [7]:
print(f"Iniciando a otimização com COBYLA para {ansatz.num_parameters} parâmetros...")

resultado_vqe = minimize(
    funcao_objetivo,
    ponto_inicial,
    args=(ansatz, hamiltoniano_qubit, estimator),
    method='COBYLA',
    tol=1e-6
)


Iniciando a otimização com COBYLA para 24 parâmetros...


## RESULTADOS

In [8]:
# A energia devolvida pelo otimizador é a energia puramente eletrônica
energia_eletronica = resultado_vqe.fun

# A energia de repulsão nuclear é uma constante clássica obtida do problema
energia_repulsao_nuclear = problema_eletronico.nuclear_repulsion_energy

# Energia total da molécula
energia_total = energia_eletronica + energia_repulsao_nuclear

print("\n" + "="*40)
print("RESULTADOS FINAIS DO VQE")
print("="*40)
print(f"Energia Eletrônica:          {energia_eletronica:.6f} Hartree")
print(f"Repulsão Nuclear:            {energia_repulsao_nuclear:.6f} Hartree")
print("-" * 40)
print(f"Energia Total do H2 (R={distancia_H2}Å): {energia_total:.6f} Hartree")
print("="*40)


RESULTADOS FINAIS DO VQE
Energia Eletrônica:          -1.857222 Hartree
Repulsão Nuclear:            0.719969 Hartree
----------------------------------------
Energia Total do H2 (R=0.735Å): -1.137253 Hartree
